# 03 (Kaggle GPU) — Model 2 (Mask R-CNN: Two-Stage CNN) & Model 3 (Mask2Former: Transformer)
**Setting Kaggle Notebook:**
- **Accelerator:** `GPU T4 x2` atau `GPU P100`
- **Internet:** `ON`

Notebook ini langsung mengunduh dataset format COCO (`Panoptic Indoor v5` + `Test Set HP Ruangan` Anda) dari Roboflow, lalu melatih dan mengevaluasi **Mask R-CNN** dan **Mask2Former**.
Semua hasil (`eval_maskrcnn.csv`, `eval_mask2former.csv`, `gpu_time_log_kaggle.csv`, dan `results.csv`) otomatis tersimpan di `/kaggle/working/cpmk3/logs/` dan `/kaggle/working/cpmk3/runs/`.


In [1]:
!pip install -q roboflow pycocotools transformers accelerate albumentations timm pandas

import os, time, json, math, glob, shutil
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils
from roboflow import Roboflow

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

# === ISI 2 BARIS INI SAJA ===
import getpass
ROBOFLOW_API_KEY = getpass.getpass('Masukkan Roboflow Private API Key: ')
CUSTOM_PROJECT   = "test-hp-ruangan"
CUSTOM_VERSION   = 2
CUSTOM_WORKSPACE = "cdaaptnia"


PROJECT_DIR = '/kaggle/working/cpmk3'
MAIN_COCO   = os.path.join(PROJECT_DIR, 'datasets', 'main_coco')
CUSTOM_COCO = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_coco')
LOGS_DIR    = os.path.join(PROJECT_DIR, 'logs')
RUNS_DIR    = os.path.join(PROJECT_DIR, 'runs')
os.makedirs(LOGS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
if not os.path.exists(MAIN_COCO):
    rf.workspace("panopticindoor").project("panoptic-indoor-segmentation").version(5).download(
        "coco-segmentation", location=MAIN_COCO, overwrite=True
    )
if not os.path.exists(CUSTOM_COCO):
    rf.workspace(CUSTOM_WORKSPACE).project(CUSTOM_PROJECT).version(CUSTOM_VERSION).download(
        "coco-segmentation", location=CUSTOM_COCO, overwrite=True
    )

def find_coco_split(root_dir, split_candidates):
    for sp in split_candidates:
        sp_dir = os.path.join(root_dir, sp)
        ann = os.path.join(sp_dir, '_annotations.coco.json')
        if os.path.exists(ann):
            return sp_dir, ann
    raise FileNotFoundError(f'Tidak menemukan _annotations.coco.json di {root_dir} ({split_candidates})')

TRAIN_DIR, TRAIN_ANN = find_coco_split(MAIN_COCO, ['train'])
VAL_DIR,   VAL_ANN   = find_coco_split(MAIN_COCO, ['valid', 'val'])
TEST_DIR,  TEST_ANN  = find_coco_split(MAIN_COCO, ['test', 'valid'])
HP_DIR,    HP_ANN    = find_coco_split(CUSTOM_COCO, ['test', 'valid', 'train'])

print('Train :', TRAIN_ANN)
print('Val   :', VAL_ANN)
print('Test  :', TEST_ANN)
print('HP    :', HP_ANN)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Device: cuda Tesla T4


Masukkan Roboflow Private API Key:  ········


Train : /kaggle/working/cpmk3/datasets/main_coco/train/_annotations.coco.json
Val   : /kaggle/working/cpmk3/datasets/main_coco/valid/_annotations.coco.json
Test  : /kaggle/working/cpmk3/datasets/main_coco/test/_annotations.coco.json
HP    : /kaggle/working/cpmk3/datasets/custom_hp_coco/test/_annotations.coco.json


## Helper: Dataset COCO & Evaluasi `pycocotools`
Memetakan kategori Roboflow secara konsisten (menghindari kategori super-parent id=0 bawaan Roboflow jika ada).


In [5]:
def get_valid_categories(ann_file):
    coco = COCO(ann_file)
    cats = coco.loadCats(coco.getCatIds())
    # Roboflow kadang menyertakan parent category dengan supercategory == 'none' dan tidak dipakai di anotasi
    used_cat_ids = set(ann['category_id'] for ann in coco.dataset['annotations'])
    valid_cats = [c for c in cats if c['id'] in used_cat_ids]
    valid_cats = sorted(valid_cats, key=lambda x: x['id'])
    cat_id_to_contig = {c['id']: i + 1 for i, c in enumerate(valid_cats)}  # 1..K (0 = background)
    contig_to_cat_id = {i + 1: c['id'] for i, c in enumerate(valid_cats)}
    id_to_name = {i + 1: c['name'] for i, c in enumerate(valid_cats)}
    return valid_cats, cat_id_to_contig, contig_to_cat_id, id_to_name

VALID_CATS, CAT2CONTIG, CONTIG2CAT, ID2NAME = get_valid_categories(TRAIN_ANN)
NUM_CLASSES = len(VALID_CATS) + 1  # +1 background untuk Mask R-CNN
print('Kelas terdeteksi:', ID2NAME, '| NUM_CLASSES (inc. bg):', NUM_CLASSES)


class CocoInstanceDataset(Dataset):
    def __init__(self, img_dir, ann_file, cat2contig, img_size=640):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.cat2contig = cat2contig
        self.img_size = img_size
        # Pastikan hanya gambar yang punya minimal 1 anotasi valid saat training
        self.img_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, info['file_name'])
        img = Image.open(img_path).convert('RGB')
        orig_w, orig_h = img.size

        img_resized = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img_tensor = torchvision.transforms.functional.to_tensor(img_resized)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes, labels, masks = [], [], []
        sx, sy = self.img_size / orig_w, self.img_size / orig_h

        for ann in anns:
            if ann['category_id'] not in self.cat2contig:
                continue
            x, y, w, h = ann['bbox']
            if w <= 1 or h <= 1:
                continue
            m = self.coco.annToMask(ann)
            m_resized = np.array(
                Image.fromarray(m).resize((self.img_size, self.img_size), Image.NEAREST),
                dtype=np.uint8,
            )
            if m_resized.sum() == 0:
                continue
            boxes.append([x * sx, y * sy, (x + w) * sx, (y + h) * sy])
            labels.append(self.cat2contig[ann['category_id']])
            masks.append(m_resized)

        if len(boxes) == 0:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
            masks_t = torch.zeros((0, self.img_size, self.img_size), dtype=torch.uint8)
        else:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
            masks_t = torch.tensor(np.stack(masks, axis=0), dtype=torch.uint8)

        target = {
            'boxes': boxes_t,
            'labels': labels_t,
            'masks': masks_t,
            'image_id': torch.tensor([img_id]),
            'orig_size': (orig_h, orig_w),
        }
        return img_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def append_gpu_log(row_dict):
    gpu_log_path = os.path.join(LOGS_DIR, 'gpu_time_log.csv')
    df_new = pd.DataFrame([row_dict])
    if os.path.exists(gpu_log_path):
        df_all = pd.concat([pd.read_csv(gpu_log_path), df_new], ignore_index=True)
    else:
        df_all = df_new
    df_all.to_csv(gpu_log_path, index=False)


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Kelas terdeteksi: {1: 'backpack', 2: 'bed', 3: 'blanket', 4: 'book', 5: 'bottle', 6: 'bowl', 7: 'box', 8: 'cabinet-merged', 9: 'cardboard', 10: 'ceiling-merged', 11: 'cell phone', 12: 'chair', 13: 'clock', 14: 'couch', 15: 'counter', 16: 'cup', 17: 'curtain', 18: 'dining table', 19: 'door-stuff', 20: 'fire-extinguisher', 21: 'floor-other-merged', 22: 'floor-wood', 23: 'food-other-merged', 24: 'fork', 25: 'handbag', 26: 'knife', 27: 'laptop', 28: 'light', 29: 'microwave', 30: 'mirror-stuff', 31: 'oven', 32: 'pan', 33: 'paper', 34: 'paper-merged', 35: 'person', 36: 'pillow', 37: 'plate', 38: 'plug', 39: 'potted plant', 40: 'refrigerator', 41: 'remote', 42: 'rug-merged', 43: 'shelf', 44: 'sink', 45: 'spoon', 46: 'st-', 47: 'stairs', 48: 'stove-merged', 49: 'table-merged', 50: 'toilet', 51: 'towel', 52: 'toy', 53: 'trash-bin', 54: 'tree-merged', 55: 'tv', 56: 'vase', 57: 'wall', 58: 'wall-brick', 59: 'wall-o

## BAGIAN 1: Training & Evaluasi Model 2 — Mask R-CNN (Two-Stage CNN)
Untuk eksperimen NRP 0, Anda bisa menjalankan `MASKRCNN_SIZES = [640]` terlebih dahulu (baseline), lalu tambah `[320, 960]` jika kuota GPU masih cukup.


In [4]:
def build_maskrcnn(num_classes, img_size=640):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=img_size,
        max_size=img_size,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model


@torch.no_grad()
def evaluate_maskrcnn_coco(model, img_dir, ann_file, img_size=640, score_thresh=0.05):
    model.eval()
    coco_gt = COCO(ann_file)
    # Petakan nama kelas di ann_file ke category_id ann_file (antisipasi perbedaan ID antara main & custom)
    name_to_ann_catid = {c['name']: c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds())}

    results_bbox, results_segm = [], []
    for img_id in coco_gt.getImgIds():
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(os.path.join(img_dir, info['file_name'])).convert('RGB')
        orig_w, orig_h = img.size
        img_resized = img.resize((img_size, img_size), Image.BILINEAR)
        tensor = torchvision.transforms.functional.to_tensor(img_resized).to(device)

        out = model([tensor])[0]
        scores = out['scores'].cpu().numpy()
        labels = out['labels'].cpu().numpy()
        boxes  = out['boxes'].cpu().numpy()
        masks  = out['masks'].cpu().numpy()  # (N, 1, H, W)

        sx, sy = orig_w / img_size, orig_h / img_size
        for s, l, b, m in zip(scores, labels, boxes, masks):
            if s < score_thresh:
                continue
            cls_name = ID2NAME.get(int(l))
            if cls_name not in name_to_ann_catid:
                continue
            cat_id = name_to_ann_catid[cls_name]
            x1, y1, x2, y2 = b[0] * sx, b[1] * sy, b[2] * sx, b[3] * sy
            results_bbox.append({
                'image_id': int(img_id),
                'category_id': int(cat_id),
                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                'score': float(s),
            })
            bin_mask = (m[0] >= 0.5).astype(np.uint8)
            bin_mask_orig = np.array(
                Image.fromarray(bin_mask).resize((orig_w, orig_h), Image.NEAREST),
                dtype=np.uint8,
            )
            rle = mask_utils.encode(np.asfortranarray(bin_mask_orig))
            rle['counts'] = rle['counts'].decode('utf-8')
            results_segm.append({
                'image_id': int(img_id),
                'category_id': int(cat_id),
                'segmentation': rle,
                'score': float(s),
            })

    metrics = {'box_mAP50': 0.0, 'box_mAP50_95': 0.0, 'mask_mAP50': 0.0, 'mask_mAP50_95': 0.0}
    if len(results_bbox) > 0:
        coco_dt = coco_gt.loadRes(results_bbox)
        ev = COCOeval(coco_gt, coco_dt, 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['box_mAP50_95'] = round(float(ev.stats[0]), 4)
        metrics['box_mAP50']    = round(float(ev.stats[1]), 4)
    if len(results_segm) > 0:
        coco_dt = coco_gt.loadRes(results_segm)
        ev = COCOeval(coco_gt, coco_dt, 'segm')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['mask_mAP50_95'] = round(float(ev.stats[0]), 4)
        metrics['mask_mAP50']    = round(float(ev.stats[1]), 4)
    return metrics


# === KONFIGURASI TRAINING MASK R-CNN ===
MASKRCNN_EPOCHS = 15
MASKRCNN_SIZES  = [640]  # Ubah ke [320, 640, 960] jika ingin uji ketiga resolusi di Mask R-CNN
MASKRCNN_BATCH  = {320: 4, 640: 4, 960: 2}

mr_eval_rows = []

for imgsz in MASKRCNN_SIZES:
    run_dir = os.path.join(RUNS_DIR, 'maskrcnn', f'maskrcnn_img{imgsz}')
    os.makedirs(run_dir, exist_ok=True)
    ckpt_path = os.path.join(run_dir, 'best.pth')
    csv_path  = os.path.join(run_dir, 'results.csv')

    train_ds = CocoInstanceDataset(TRAIN_DIR, TRAIN_ANN, CAT2CONTIG, img_size=imgsz)
    train_loader = DataLoader(train_ds, batch_size=MASKRCNN_BATCH[imgsz], shuffle=True,
                              num_workers=2, collate_fn=collate_fn)

    model = build_maskrcnn(NUM_CLASSES, img_size=imgsz).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    if not os.path.exists(ckpt_path):
        t0 = time.time()
        history = []
        best_loss = float('inf')

        for epoch in range(1, MASKRCNN_EPOCHS + 1):
            model.train()
            ep_loss = 0.0
            n_steps = 0
            ep_t0 = time.time()
            for imgs, targets in train_loader:
                # Lewati batch yang tidak memiliki objek
                valid_pairs = [(im.to(device), {k: v.to(device) if torch.is_tensor(v) else v for k, v in tg.items()})
                               for im, tg in zip(imgs, targets) if len(tg['boxes']) > 0]
                if not valid_pairs:
                    continue
                im_list, tg_list = zip(*valid_pairs)

                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    loss_dict = model(list(im_list), list(tg_list))
                    losses = sum(loss for loss in loss_dict.values())

                scaler.scale(losses).backward()
                scaler.step(optimizer)
                scaler.update()

                ep_loss += float(losses.item())
                n_steps += 1

            avg_loss = ep_loss / max(n_steps, 1)
            ep_sec = round(time.time() - ep_t0, 1)
            print(f'[Mask R-CNN img={imgsz}] Epoch {epoch}/{MASKRCNN_EPOCHS} | loss={avg_loss:.4f} | {ep_sec}s')
            history.append({'epoch': epoch, 'train_loss': round(avg_loss, 4), 'epoch_sec': ep_sec})

            if avg_loss < best_loss:
                best_loss = avg_loss
                torch.save(model.state_dict(), ckpt_path)

        pd.DataFrame(history).to_csv(csv_path, index=False)
        elapsed_min = round((time.time() - t0) / 60.0, 2)
        append_gpu_log({
            'model': 'MaskRCNN-R50-FPNv2',
            'architecture_family': 'Two-Stage CNN',
            'imgsz': imgsz,
            'epochs': MASKRCNN_EPOCHS,
            'batch_size': MASKRCNN_BATCH[imgsz],
            'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
            'train_time_minutes': elapsed_min,
            'results_csv': csv_path,
        })
    else:
        print(f'[SKIP TRAIN] Memuat checkpoint tersimpan: {ckpt_path}')

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    for split_label, d_dir, d_ann in [('Test_Asli', TEST_DIR, TEST_ANN), ('Test_HP_Sendiri', HP_DIR, HP_ANN)]:
        t0 = time.time()
        m = evaluate_maskrcnn_coco(model, d_dir, d_ann, img_size=imgsz)
        m.update({
            'model': 'MaskRCNN-R50-FPNv2',
            'architecture_family': 'Two-Stage CNN',
            'imgsz': imgsz,
            'dataset': split_label,
            'eval_time_sec': round(time.time() - t0, 2),
        })
        mr_eval_rows.append(m)

    del model
    torch.cuda.empty_cache()

df_mr = pd.DataFrame(mr_eval_rows)
df_mr.to_csv(os.path.join(LOGS_DIR, 'eval_maskrcnn.csv'), index=False)
df_mr


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth


100%|██████████| 177M/177M [00:00<00:00, 210MB/s] 
`torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


[Mask R-CNN img=640] Epoch 1/15 | loss=2.3536 | 53.7s
[Mask R-CNN img=640] Epoch 2/15 | loss=1.5041 | 51.8s
[Mask R-CNN img=640] Epoch 3/15 | loss=1.1829 | 56.5s
[Mask R-CNN img=640] Epoch 4/15 | loss=0.9919 | 55.4s
[Mask R-CNN img=640] Epoch 5/15 | loss=0.8473 | 56.1s
[Mask R-CNN img=640] Epoch 6/15 | loss=0.7512 | 55.4s
[Mask R-CNN img=640] Epoch 7/15 | loss=0.6592 | 55.6s
[Mask R-CNN img=640] Epoch 8/15 | loss=0.5908 | 55.8s
[Mask R-CNN img=640] Epoch 9/15 | loss=0.5576 | 55.9s
[Mask R-CNN img=640] Epoch 10/15 | loss=0.5036 | 55.6s
[Mask R-CNN img=640] Epoch 11/15 | loss=0.4593 | 55.8s
[Mask R-CNN img=640] Epoch 12/15 | loss=0.4298 | 55.7s
[Mask R-CNN img=640] Epoch 13/15 | loss=0.4175 | 56.2s
[Mask R-CNN img=640] Epoch 14/15 | loss=0.3955 | 55.5s
[Mask R-CNN img=640] Epoch 15/15 | loss=0.3652 | 55.6s
loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image ev

,box_mAP50,box_mAP50_95,mask_mAP50,mask_mAP50_95,model,architecture_family,imgsz,dataset,eval_time_sec
0,0.1539,0.0889,0.1448,0.0865,MaskRCNN-R50-FPNv2,Two-Stage CNN,640,Test_Asli,141.46
1,0.1363,0.0915,0.1254,0.0783,MaskRCNN-R50-FPNv2,Two-Stage CNN,640,Test_HP_Sendiri,12.07


## BAGIAN 2: Training & Evaluasi Model 3 — Mask2Former (Transformer)
Menggunakan `facebook/mask2former-swin-tiny-coco-instance` dengan `fp16` dan `batch_size=2` (+ gradient accumulation 4 langkah -> efektif batch 8) agar hemat VRAM di GPU T4 Colab Free.


In [6]:
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

M2F_MODEL_ID = 'facebook/mask2former-swin-tiny-coco-instance'
M2F_EPOCHS   = 15
M2F_IMG_SIZE = 640
M2F_BATCH    = 2
GRAD_ACCUM   = 4

M2F_ID2LABEL   = {i: c['name'] for i, c in enumerate(VALID_CATS)}
M2F_LABEL2ID   = {c['name']: i for i, c in enumerate(VALID_CATS)}
M2F_CAT2CONTIG = {c['id']: i for i, c in enumerate(VALID_CATS)}

processor = AutoImageProcessor.from_pretrained(
    M2F_MODEL_ID,
    size={'height': M2F_IMG_SIZE, 'width': M2F_IMG_SIZE},
)


class Mask2FormerCocoDataset(Dataset):
    def __init__(self, img_dir, ann_file, cat2contig, img_size=640):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.cat2contig = cat2contig
        self.img_size = img_size
        self.img_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')

        masks_list, classes_list = [], []
        for ann in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id)):
            if ann['category_id'] not in self.cat2contig:
                continue
            m = self.coco.annToMask(ann)
            if m.sum() == 0:
                continue
            m_res = np.array(
                Image.fromarray(m.astype(np.uint8)).resize((self.img_size, self.img_size), Image.NEAREST),
                dtype=np.float32,
            )
            if m_res.sum() == 0:
                continue
            masks_list.append(m_res)
            classes_list.append(self.cat2contig[ann['category_id']])

        if len(masks_list) == 0:
            return None

        inputs = processor(images=img, return_tensors='pt')
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'pixel_mask': inputs['pixel_mask'].squeeze(0),
            'mask_labels': torch.tensor(np.stack(masks_list, axis=0), dtype=torch.float32),
            'class_labels': torch.tensor(classes_list, dtype=torch.int64),
        }


def m2f_collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'pixel_mask': torch.stack([b['pixel_mask'] for b in batch]),
        'mask_labels': [b['mask_labels'] for b in batch],
        'class_labels': [b['class_labels'] for b in batch],
    }


m2f_run_dir  = os.path.join(RUNS_DIR, 'mask2former', f'm2f_swin_tiny_img{M2F_IMG_SIZE}')
os.makedirs(m2f_run_dir, exist_ok=True)
m2f_ckpt_dir = os.path.join(m2f_run_dir, 'best_model')
m2f_csv_path = os.path.join(m2f_run_dir, 'results.csv')

if not os.path.exists(m2f_ckpt_dir):
    m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained(
        M2F_MODEL_ID,
        id2label=M2F_ID2LABEL,
        label2id=M2F_LABEL2ID,
        ignore_mismatched_sizes=True,
    ).to(device)

    train_ds = Mask2FormerCocoDataset(TRAIN_DIR, TRAIN_ANN, M2F_CAT2CONTIG, img_size=M2F_IMG_SIZE)
    train_loader = DataLoader(train_ds, batch_size=M2F_BATCH, shuffle=True,
                              num_workers=2, collate_fn=m2f_collate_fn)

    optimizer = torch.optim.AdamW(m2f_model.parameters(), lr=5e-5, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    t0 = time.time()
    history = []
    best_loss = float('inf')

    for epoch in range(1, M2F_EPOCHS + 1):
        m2f_model.train()
        ep_loss, n_steps = 0.0, 0
        ep_t0 = time.time()
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader, 1):
            if batch is None:
                continue
            pixel_values = batch['pixel_values'].to(device)
            pixel_mask   = batch['pixel_mask'].to(device)
            mask_labels  = [m.to(device) for m in batch['mask_labels']]
            class_labels = [c.to(device) for c in batch['class_labels']]

            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                outputs = m2f_model(
                    pixel_values=pixel_values,
                    pixel_mask=pixel_mask,
                    mask_labels=mask_labels,
                    class_labels=class_labels,
                )
                loss = outputs.loss / GRAD_ACCUM

            scaler.scale(loss).backward()
            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            ep_loss += float(outputs.loss.item())
            n_steps += 1

        avg_loss = ep_loss / max(n_steps, 1)
        ep_sec = round(time.time() - ep_t0, 1)
        print(f'[Mask2Former img={M2F_IMG_SIZE}] Epoch {epoch}/{M2F_EPOCHS} | loss={avg_loss:.4f} | {ep_sec}s')
        history.append({'epoch': epoch, 'train_loss': round(avg_loss, 4), 'epoch_sec': ep_sec})

        if avg_loss < best_loss:
            best_loss = avg_loss
            m2f_model.save_pretrained(m2f_ckpt_dir)

    pd.DataFrame(history).to_csv(m2f_csv_path, index=False)
    elapsed_min = round((time.time() - t0) / 60.0, 2)
    append_gpu_log({
        'model': 'Mask2Former-Swin-Tiny',
        'architecture_family': 'Transformer',
        'imgsz': M2F_IMG_SIZE,
        'epochs': M2F_EPOCHS,
        'batch_size': M2F_BATCH * GRAD_ACCUM,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'train_time_minutes': elapsed_min,
        'results_csv': m2f_csv_path,
    })
else:
    print(f'[SKIP TRAIN] Memuat Mask2Former tersimpan: {m2f_ckpt_dir}')
    m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained(m2f_ckpt_dir).to(device)


@torch.no_grad()
def evaluate_mask2former_coco(model, img_dir, ann_file, score_thresh=0.05):
    model.eval()
    coco_gt = COCO(ann_file)
    name_to_ann_catid = {c['name']: c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds())}
    results_bbox, results_segm = [], []

    for img_id in coco_gt.getImgIds():
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(os.path.join(img_dir, info['file_name'])).convert('RGB')
        orig_w, orig_h = img.size
        inputs = processor(images=img, return_tensors='pt').to(device)
        outputs = model(**inputs)
        pred = processor.post_process_instance_segmentation(
            outputs, threshold=score_thresh, target_sizes=[(orig_h, orig_w)]
        )[0]
        seg_map = pred['segmentation'].cpu().numpy()
        for seg_info in pred['segments_info']:
            cls_name = M2F_ID2LABEL.get(int(seg_info['label_id']))
            if cls_name not in name_to_ann_catid:
                continue
            cat_id = name_to_ann_catid[cls_name]
            bin_mask = (seg_map == seg_info['id']).astype(np.uint8)
            if bin_mask.sum() == 0:
                continue
            ys, xs = np.where(bin_mask == 1)
            x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
            score = float(seg_info['score'])
            results_bbox.append({
                'image_id': int(img_id), 'category_id': int(cat_id),
                'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': score,
            })
            rle = mask_utils.encode(np.asfortranarray(bin_mask))
            rle['counts'] = rle['counts'].decode('utf-8')
            results_segm.append({
                'image_id': int(img_id), 'category_id': int(cat_id),
                'segmentation': rle, 'score': score,
            })

    metrics = {'box_mAP50': 0.0, 'box_mAP50_95': 0.0, 'mask_mAP50': 0.0, 'mask_mAP50_95': 0.0}
    if len(results_bbox) > 0:
        ev = COCOeval(coco_gt, coco_gt.loadRes(results_bbox), 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['box_mAP50_95'] = round(float(ev.stats[0]), 4)
        metrics['box_mAP50']    = round(float(ev.stats[1]), 4)
    if len(results_segm) > 0:
        ev = COCOeval(coco_gt, coco_gt.loadRes(results_segm), 'segm')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['mask_mAP50_95'] = round(float(ev.stats[0]), 4)
        metrics['mask_mAP50']    = round(float(ev.stats[1]), 4)
    return metrics


m2f_eval_rows = []
for split_label, d_dir, d_ann in [('Test_Asli', TEST_DIR, TEST_ANN), ('Test_HP_Sendiri', HP_DIR, HP_ANN)]:
    t0 = time.time()
    m = evaluate_mask2former_coco(m2f_model, d_dir, d_ann)
    m.update({
        'model': 'Mask2Former-Swin-Tiny',
        'architecture_family': 'Transformer',
        'imgsz': M2F_IMG_SIZE,
        'dataset': split_label,
        'eval_time_sec': round(time.time() - t0, 2),
    })
    m2f_eval_rows.append(m)

df_m2f = pd.DataFrame(m2f_eval_rows)
df_m2f.to_csv(os.path.join(LOGS_DIR, 'eval_mask2former.csv'), index=False)
df_m2f

Loading weights:   0%|          | 0/554 [00:00<?, ?it/s]

[transformers] Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                                                                                                                                | Status     |                                                                                         
-----------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.pixel_level_module.encoder.swin.encoder.layers.{0, 1, 2, 3}.blocks.{0, 1, 2, 3, 4, 5}.attention.self.relative_position_index | UNEXPECTED |                                                                                         
model.pixel_level_module.encoder.swin.layernorm.weight                                                                             | MISSING    |                                    

loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
[Mask2Former img=640] Epoch 1/15 | loss=80.1460 | 108.5s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[Mask2Former img=640] Epoch 2/15 | loss=59.1203 | 107.5s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (20 outputs left out)

[Mask2Former img=640] Epoch 15/15 | loss=22.1885 | 107.7s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.97s).
Accumulating evaluation results...
DONE (t=0.56s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.060
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.097
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.057
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.024
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.048
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.089
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.077
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.136
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

,box_mAP50,box_mAP50_95,mask_mAP50,mask_mAP50_95,model,architecture_family,imgsz,dataset,eval_time_sec
0,0.0973,0.0597,0.0683,0.0455,Mask2Former-Swin-Tiny,Transformer,640,Test_Asli,147.83
1,0.0942,0.0489,0.0411,0.0264,Mask2Former-Swin-Tiny,Transformer,640,Test_HP_Sendiri,14.63


## BAGIAN 3 (NRP 0): Eksperimen Ukuran Input `320` & `960` — Mask R-CNN
> **Cara menjalankan tanpa mengulang `640px`:** Jalankan **Cell 1** & **Cell 3 (Helper COCO)**, lalu langsung lompat ke cell ini dan Bagian 4 (tidak perlu jalankan ulang Bagian 1 & 2 yang `640px`).


In [6]:
# Pastikan fungsi build_maskrcnn & evaluate_maskrcnn_coco tersedia tanpa harus menjalankan ulang Bagian 1 (640px)
def build_maskrcnn(num_classes, img_size=640):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=img_size,
        max_size=img_size,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model


@torch.no_grad()
def evaluate_maskrcnn_coco(model, img_dir, ann_file, img_size=640, score_thresh=0.05):
    model.eval()
    coco_gt = COCO(ann_file)
    name_to_ann_catid = {c['name']: c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds())}
    results_bbox, results_segm = [], []
    for img_id in coco_gt.getImgIds():
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(os.path.join(img_dir, info['file_name'])).convert('RGB')
        orig_w, orig_h = img.size
        tensor = torchvision.transforms.functional.to_tensor(
            img.resize((img_size, img_size), Image.BILINEAR)
        ).to(device)
        out = model([tensor])[0]
        sx, sy = orig_w / img_size, orig_h / img_size
        for s, l, b, m in zip(out['scores'].cpu().numpy(), out['labels'].cpu().numpy(),
                              out['boxes'].cpu().numpy(), out['masks'].cpu().numpy()):
            if s < score_thresh:
                continue
            cls_name = ID2NAME.get(int(l))
            if cls_name not in name_to_ann_catid:
                continue
            cat_id = name_to_ann_catid[cls_name]
            x1, y1, x2, y2 = b[0] * sx, b[1] * sy, b[2] * sx, b[3] * sy
            results_bbox.append({
                'image_id': int(img_id), 'category_id': int(cat_id),
                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], 'score': float(s),
            })
            bin_mask = np.array(
                Image.fromarray((m[0] >= 0.5).astype(np.uint8)).resize((orig_w, orig_h), Image.NEAREST),
                dtype=np.uint8,
            )
            rle = mask_utils.encode(np.asfortranarray(bin_mask))
            rle['counts'] = rle['counts'].decode('utf-8')
            results_segm.append({
                'image_id': int(img_id), 'category_id': int(cat_id),
                'segmentation': rle, 'score': float(s),
            })
    metrics = {'box_mAP50': 0.0, 'box_mAP50_95': 0.0, 'mask_mAP50': 0.0, 'mask_mAP50_95': 0.0}
    if results_bbox:
        ev = COCOeval(coco_gt, coco_gt.loadRes(results_bbox), 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['box_mAP50_95'], metrics['box_mAP50'] = round(float(ev.stats[0]), 4), round(float(ev.stats[1]), 4)
    if results_segm:
        ev = COCOeval(coco_gt, coco_gt.loadRes(results_segm), 'segm')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['mask_mAP50_95'], metrics['mask_mAP50'] = round(float(ev.stats[0]), 4), round(float(ev.stats[1]), 4)
    return metrics


MR_EPOCHS_EXTRA = 15
MR_EXTRA_CFG = {320: 4, 960: 2}
mr_extra_rows = []

for imgsz, bsz in MR_EXTRA_CFG.items():
    run_dir = os.path.join(RUNS_DIR, 'maskrcnn', f'maskrcnn_img{imgsz}')
    os.makedirs(run_dir, exist_ok=True)
    ckpt_path = os.path.join(run_dir, 'best.pth')
    csv_path  = os.path.join(run_dir, 'results.csv')

    train_ds = CocoInstanceDataset(TRAIN_DIR, TRAIN_ANN, CAT2CONTIG, img_size=imgsz)
    train_loader = DataLoader(train_ds, batch_size=bsz, shuffle=True, num_workers=2, collate_fn=collate_fn)
    model = build_maskrcnn(NUM_CLASSES, img_size=imgsz).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

    if not os.path.exists(ckpt_path):
        t0, history, best_loss = time.time(), [], float('inf')
        for epoch in range(1, MR_EPOCHS_EXTRA + 1):
            model.train()
            ep_loss, n_steps, ep_t0 = 0.0, 0, time.time()
            for imgs, targets in train_loader:
                valid_pairs = [(im.to(device), {k: v.to(device) if torch.is_tensor(v) else v for k, v in tg.items()})
                               for im, tg in zip(imgs, targets) if len(tg['boxes']) > 0]
                if not valid_pairs:
                    continue
                im_list, tg_list = zip(*valid_pairs)
                optimizer.zero_grad()
                with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                    losses = sum(model(list(im_list), list(tg_list)).values())
                scaler.scale(losses).backward()
                scaler.step(optimizer)
                scaler.update()
                ep_loss += float(losses.item()); n_steps += 1
            avg_loss = ep_loss / max(n_steps, 1)
            ep_sec = round(time.time() - ep_t0, 1)
            print(f'[Mask R-CNN img={imgsz}] Epoch {epoch}/{MR_EPOCHS_EXTRA} | loss={avg_loss:.4f} | {ep_sec}s')
            history.append({'epoch': epoch, 'train_loss': round(avg_loss, 4), 'epoch_sec': ep_sec})
            if avg_loss < best_loss:
                best_loss = avg_loss
                torch.save(model.state_dict(), ckpt_path)
        pd.DataFrame(history).to_csv(csv_path, index=False)
        append_gpu_log({
            'model': 'MaskRCNN-R50-FPNv2', 'architecture_family': 'Two-Stage CNN',
            'imgsz': imgsz, 'epochs': MR_EPOCHS_EXTRA, 'batch_size': bsz,
            'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
            'train_time_minutes': round((time.time() - t0) / 60.0, 2),
            'results_csv': csv_path,
        })
    else:
        print(f'[SKIP TRAIN] Memuat checkpoint tersimpan: {ckpt_path}')

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    for split_label, d_dir, d_ann in [('Test_Asli', TEST_DIR, TEST_ANN), ('Test_HP_Sendiri', HP_DIR, HP_ANN)]:
        t_e = time.time()
        m = evaluate_maskrcnn_coco(model, d_dir, d_ann, img_size=imgsz)
        m.update({
            'model': 'MaskRCNN-R50-FPNv2', 'architecture_family': 'Two-Stage CNN',
            'imgsz': imgsz, 'dataset': split_label, 'eval_time_sec': round(time.time() - t_e, 2),
        })
        mr_extra_rows.append(m)
    del model; torch.cuda.empty_cache()

df_mr_extra = pd.DataFrame(mr_extra_rows)
df_mr_extra.to_csv(os.path.join(LOGS_DIR, 'eval_maskrcnn_320_960.csv'), index=False)
df_mr_extra


loading annotations into memory...
Done (t=0.04s)
creating index...
index created!
Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_v2_coco-73cbd019.pth


100%|██████████| 177M/177M [00:00<00:00, 254MB/s] 


[Mask R-CNN img=320] Epoch 1/15 | loss=2.5251 | 37.9s
[Mask R-CNN img=320] Epoch 2/15 | loss=1.7166 | 33.7s
[Mask R-CNN img=320] Epoch 3/15 | loss=1.3940 | 35.9s
[Mask R-CNN img=320] Epoch 4/15 | loss=1.1547 | 37.6s
[Mask R-CNN img=320] Epoch 5/15 | loss=1.0011 | 36.5s
[Mask R-CNN img=320] Epoch 6/15 | loss=0.8788 | 37.2s
[Mask R-CNN img=320] Epoch 7/15 | loss=0.7741 | 37.0s
[Mask R-CNN img=320] Epoch 8/15 | loss=0.6970 | 36.9s
[Mask R-CNN img=320] Epoch 9/15 | loss=0.6509 | 37.2s
[Mask R-CNN img=320] Epoch 10/15 | loss=0.6039 | 37.1s
[Mask R-CNN img=320] Epoch 11/15 | loss=0.5655 | 37.0s
[Mask R-CNN img=320] Epoch 12/15 | loss=0.5255 | 37.1s
[Mask R-CNN img=320] Epoch 13/15 | loss=0.4879 | 36.8s
[Mask R-CNN img=320] Epoch 14/15 | loss=0.4474 | 37.1s
[Mask R-CNN img=320] Epoch 15/15 | loss=0.4429 | 37.0s
loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.03s)
creating index...
index created!
Running per image ev

,box_mAP50,box_mAP50_95,mask_mAP50,mask_mAP50_95,model,architecture_family,imgsz,dataset,eval_time_sec
0,0.1251,0.0691,0.1136,0.0648,MaskRCNN-R50-FPNv2,Two-Stage CNN,320,Test_Asli,104.51
1,0.0828,0.0489,0.0745,0.0438,MaskRCNN-R50-FPNv2,Two-Stage CNN,320,Test_HP_Sendiri,8.11
2,0.1348,0.0754,0.1267,0.0747,MaskRCNN-R50-FPNv2,Two-Stage CNN,960,Test_Asli,222.32
3,0.1356,0.0806,0.1260,0.0795,MaskRCNN-R50-FPNv2,Two-Stage CNN,960,Test_HP_Sendiri,21.20


## BAGIAN 4 (NRP 0): Eksperimen Ukuran Input `320` & `960` — Mask2Former (Transformer)
Menggunakan `batch=2, accum=4` pada `320px` dan `batch=1, accum=8` pada `960px` (batch efektif sama-sama 8, aman dari OOM di Tesla T4 15GB).


In [7]:
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

M2F_MODEL_ID = 'facebook/mask2former-swin-tiny-coco-instance'
M2F_EPOCHS_EXTRA = 15
M2F_EXTRA_CFG = {
    320: {'batch': 2, 'accum': 4},
    960: {'batch': 1, 'accum': 8},
}

M2F_ID2LABEL   = {i: c['name'] for i, c in enumerate(VALID_CATS)}
M2F_LABEL2ID   = {c['name']: i for i, c in enumerate(VALID_CATS)}
M2F_CAT2CONTIG = {c['id']: i for i, c in enumerate(VALID_CATS)}


class Mask2FormerCocoDatasetV2(Dataset):
    def __init__(self, img_dir, ann_file, cat2contig, processor, img_size):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.cat2contig = cat2contig
        self.processor = processor
        self.img_size = img_size
        self.img_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')
        masks_list, classes_list = [], []
        for ann in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id)):
            if ann['category_id'] not in self.cat2contig:
                continue
            m = self.coco.annToMask(ann)
            if m.sum() == 0:
                continue
            m_res = np.array(
                Image.fromarray(m.astype(np.uint8)).resize((self.img_size, self.img_size), Image.NEAREST),
                dtype=np.float32,
            )
            if m_res.sum() == 0:
                continue
            masks_list.append(m_res)
            classes_list.append(self.cat2contig[ann['category_id']])
        if not masks_list:
            return None
        inp = self.processor(images=img, return_tensors='pt')
        return {
            'pixel_values': inp['pixel_values'].squeeze(0),
            'pixel_mask': inp['pixel_mask'].squeeze(0),
            'mask_labels': torch.tensor(np.stack(masks_list, axis=0), dtype=torch.float32),
            'class_labels': torch.tensor(classes_list, dtype=torch.int64),
        }


def m2f_collate_v2(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'pixel_mask': torch.stack([b['pixel_mask'] for b in batch]),
        'mask_labels': [b['mask_labels'] for b in batch],
        'class_labels': [b['class_labels'] for b in batch],
    }


@torch.no_grad()
def evaluate_m2f_v2(model, processor, img_dir, ann_file, score_thresh=0.05):
    model.eval()
    coco_gt = COCO(ann_file)
    name_to_id = {c['name']: c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds())}
    res_b, res_s = [], []
    for img_id in coco_gt.getImgIds():
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(os.path.join(img_dir, info['file_name'])).convert('RGB')
        ow, oh = img.size
        inp = processor(images=img, return_tensors='pt').to(device)
        out = model(**inp)
        pred = processor.post_process_instance_segmentation(out, threshold=score_thresh, target_sizes=[(oh, ow)])[0]
        seg_map = pred['segmentation'].cpu().numpy()
        for si in pred['segments_info']:
            cname = M2F_ID2LABEL.get(int(si['label_id']))
            if cname not in name_to_id:
                continue
            cid = name_to_id[cname]
            bm = (seg_map == si['id']).astype(np.uint8)
            if bm.sum() == 0:
                continue
            ys, xs = np.where(bm == 1)
            x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
            sc = float(si['score'])
            res_b.append({'image_id': int(img_id), 'category_id': int(cid),
                          'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': sc})
            rle = mask_utils.encode(np.asfortranarray(bm))
            rle['counts'] = rle['counts'].decode('utf-8')
            res_s.append({'image_id': int(img_id), 'category_id': int(cid), 'segmentation': rle, 'score': sc})
    met = {'box_mAP50': 0.0, 'box_mAP50_95': 0.0, 'mask_mAP50': 0.0, 'mask_mAP50_95': 0.0}
    if res_b:
        ev = COCOeval(coco_gt, coco_gt.loadRes(res_b), 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        met['box_mAP50_95'], met['box_mAP50'] = round(float(ev.stats[0]), 4), round(float(ev.stats[1]), 4)
    if res_s:
        ev = COCOeval(coco_gt, coco_gt.loadRes(res_s), 'segm')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        met['mask_mAP50_95'], met['mask_mAP50'] = round(float(ev.stats[0]), 4), round(float(ev.stats[1]), 4)
    return met


m2f_extra_rows = []
for imgsz, cfg in M2F_EXTRA_CFG.items():
    bsz, acc = cfg['batch'], cfg['accum']
    run_dir  = os.path.join(RUNS_DIR, 'mask2former', f'm2f_swin_tiny_img{imgsz}')
    os.makedirs(run_dir, exist_ok=True)
    ckpt_dir = os.path.join(run_dir, 'best_model')
    csv_path = os.path.join(run_dir, 'results.csv')

    proc = AutoImageProcessor.from_pretrained(M2F_MODEL_ID, size={'height': imgsz, 'width': imgsz})
    if not os.path.exists(ckpt_dir):
        model = Mask2FormerForUniversalSegmentation.from_pretrained(
            M2F_MODEL_ID, id2label=M2F_ID2LABEL, label2id=M2F_LABEL2ID, ignore_mismatched_sizes=True
        ).to(device)
        loader = DataLoader(Mask2FormerCocoDatasetV2(TRAIN_DIR, TRAIN_ANN, M2F_CAT2CONTIG, proc, imgsz),
                            batch_size=bsz, shuffle=True, num_workers=2, collate_fn=m2f_collate_v2)
        opt = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
        scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())
        t0, hist, best_loss = time.time(), [], float('inf')
        for ep in range(1, M2F_EPOCHS_EXTRA + 1):
            model.train()
            ep_loss, n_st, ep_t0 = 0.0, 0, time.time()
            opt.zero_grad()
            for st, batch in enumerate(loader, 1):
                if batch is None:
                    continue
                with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                    out = model(
                        pixel_values=batch['pixel_values'].to(device),
                        pixel_mask=batch['pixel_mask'].to(device),
                        mask_labels=[m.to(device) for m in batch['mask_labels']],
                        class_labels=[c.to(device) for c in batch['class_labels']],
                    )
                    loss = out.loss / acc
                scaler.scale(loss).backward()
                if st % acc == 0 or st == len(loader):
                    scaler.step(opt); scaler.update(); opt.zero_grad()
                ep_loss += float(out.loss.item()); n_st += 1
            avg_l = ep_loss / max(n_st, 1)
            ep_s = round(time.time() - ep_t0, 1)
            print(f'[Mask2Former img={imgsz}] Epoch {ep}/{M2F_EPOCHS_EXTRA} | loss={avg_l:.4f} | {ep_s}s')
            hist.append({'epoch': ep, 'train_loss': round(avg_l, 4), 'epoch_sec': ep_s})
            if avg_l < best_loss:
                best_loss = avg_l
                model.save_pretrained(ckpt_dir)
        pd.DataFrame(hist).to_csv(csv_path, index=False)
        append_gpu_log({
            'model': 'Mask2Former-Swin-Tiny', 'architecture_family': 'Transformer',
            'imgsz': imgsz, 'epochs': M2F_EPOCHS_EXTRA, 'batch_size': bsz * acc,
            'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
            'train_time_minutes': round((time.time() - t0) / 60.0, 2),
            'results_csv': csv_path,
        })
    else:
        print(f'[SKIP TRAIN] Memuat checkpoint tersimpan: {ckpt_dir}')

    model = Mask2FormerForUniversalSegmentation.from_pretrained(ckpt_dir).to(device)
    for split_label, d_dir, d_ann in [('Test_Asli', TEST_DIR, TEST_ANN), ('Test_HP_Sendiri', HP_DIR, HP_ANN)]:
        t_e = time.time()
        m = evaluate_m2f_v2(model, proc, d_dir, d_ann)
        m.update({
            'model': 'Mask2Former-Swin-Tiny', 'architecture_family': 'Transformer',
            'imgsz': imgsz, 'dataset': split_label, 'eval_time_sec': round(time.time() - t_e, 2),
        })
        m2f_extra_rows.append(m)
    del model; torch.cuda.empty_cache()

df_m2f_extra = pd.DataFrame(m2f_extra_rows)
df_m2f_extra.to_csv(os.path.join(LOGS_DIR, 'eval_mask2former_320_960.csv'), index=False)
df_m2f_extra


preprocessor_config.json:   0%|          | 0.00/537 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/79.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  190MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/554 [00:00<?, ?it/s]

[transformers] Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                                                                                                                                | Status     |                                                                                         
-----------------------------------------------------------------------------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
model.pixel_level_module.encoder.swin.encoder.layers.{0, 1, 2, 3}.blocks.{0, 1, 2, 3, 4, 5}.attention.self.relative_position_index | UNEXPECTED |                                                                                         
model.pixel_level_module.encoder.swin.layernorm.weight                                                                             | MISSING    |                                    

... (61 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/556 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.06s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.02s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.63s).
Accumulating evaluation results...
DONE (t=0.55s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.061
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.092
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.061
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.020
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.069
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.085
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.079
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.136
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets

,box_mAP50,box_mAP50_95,mask_mAP50,mask_mAP50_95,model,architecture_family,imgsz,dataset,eval_time_sec
0,0.0871,0.0523,0.0687,0.0412,Mask2Former-Swin-Tiny,Transformer,320,Test_Asli,105.31
1,0.0481,0.0348,0.0369,0.0214,Mask2Former-Swin-Tiny,Transformer,320,Test_HP_Sendiri,9.11
2,0.0920,0.0609,0.0641,0.0440,Mask2Former-Swin-Tiny,Transformer,960,Test_Asli,220.95
3,0.0771,0.0488,0.0531,0.0284,Mask2Former-Swin-Tiny,Transformer,960,Test_HP_Sendiri,23.58


## Download Semua Hasil (ZIP)
Jalankan cell ini di akhir agar semua file `.csv` (`eval_*.csv`, `gpu_time_log.csv`, dan `results.csv`) terpaket ke `kaggle_results_cp3_all.zip`.


In [8]:
import zipfile
from IPython.display import FileLink

zip_path = '/kaggle/working/kaggle_results_cp3_all.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for root, _, files in os.walk(PROJECT_DIR):
        for fn in files:
            if fn.endswith('.csv'):
                full = os.path.join(root, fn)
                zf.write(full, arcname=os.path.relpath(full, PROJECT_DIR))
print('Siap download:', zip_path)
FileLink('kaggle_results_cp3_all.zip')


Siap download: /kaggle/working/kaggle_results_cp3_all.zip


/kaggle/working/kaggle_results_cp3_all.zip